In [1]:
# Basic libraries
import os
import json
import re
import string
import pickle
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

# Text preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# CNN feature extraction
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model

# Example paths for MS COCO subset
CAPTIONS_FILE = '/content/annotations/captions_train2017.json'
IMAGES_DIR = '/content/train2017'

# Files to save preprocessing outputs
FEATURES_FILE = '/content/coco_image_features.pkl'
TOKENIZER_FILE = '/content/tokenizer.pkl'
MAPPINGS_FILE = '/content/vocab_mappings.pkl'


In [2]:
# Load MS COCO captions file
with open(CAPTIONS_FILE, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

# Build image_id -> file_name mapping
image_id_to_file = {img['id']: img['file_name'] for img in coco_data['images']}

# Collect captions grouped by image file name
captions_mapping = {}
for ann in coco_data['annotations']:
    image_id = ann['image_id']
    file_name = image_id_to_file[image_id]
    caption = ann['caption']
    captions_mapping.setdefault(file_name, []).append(caption)

print('Number of images with captions:', len(captions_mapping))
sample_file = next(iter(captions_mapping))
print('Sample image:', sample_file)
print('Sample captions:', captions_mapping[sample_file][:2])


FileNotFoundError: [Errno 2] No such file or directory: '/content/annotations/captions_train2017.json'

In [ ]:
# Clean each caption in a simple standard way
def clean_caption(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep letters and spaces only
    text = re.sub(r'\s+', ' ', text).strip()
    words = [word for word in text.split() if len(word) > 1]
    return ' '.join(words)

# Add start/end tokens for sequence generation
cleaned_captions_mapping = {}
all_cleaned_captions = []

for file_name, captions in captions_mapping.items():
    cleaned_list = []
    for caption in captions:
        cleaned = clean_caption(caption)
        cleaned = '<start> ' + cleaned + ' <end>'
        cleaned_list.append(cleaned)
        all_cleaned_captions.append(cleaned)
    cleaned_captions_mapping[file_name] = cleaned_list

print('Total cleaned captions:', len(all_cleaned_captions))
print('Example cleaned caption:', all_cleaned_captions[0])


In [ ]:
# Tokenize processed captions
tokenizer = Tokenizer(oov_token='<unk>', filters='')
tokenizer.fit_on_texts(all_cleaned_captions)

# Convert captions into token sequences
caption_sequences = tokenizer.texts_to_sequences(all_cleaned_captions)

print('Vocabulary size from tokenizer:', len(tokenizer.word_index) + 1)
print('Sample tokenized caption:', caption_sequences[0])


In [ ]:
# Build vocabulary from processed captions only
word_counts = Counter()
for caption in all_cleaned_captions:
    word_counts.update(caption.split())

# Keep all words found in the processed captions
vocab = sorted(word_counts.keys())

# Create mappings
word_to_index = tokenizer.word_index.copy()
index_to_word = {index: word for word, index in word_to_index.items()}

vocab_size = len(word_to_index) + 1  # +1 because index 0 is padding

print('Vocabulary size:', vocab_size)
print('First 15 vocabulary words:', vocab[:15])
print('Index of <start>:', word_to_index.get('<start>'))
print('Index of <end>:', word_to_index.get('<end>'))


In [ ]:
# Save tokenizer and mappings for later use
with open(TOKENIZER_FILE, 'wb') as f:
    pickle.dump(tokenizer, f)

with open(MAPPINGS_FILE, 'wb') as f:
    pickle.dump({
        'word_to_index': word_to_index,
        'index_to_word': index_to_word,
        'vocab_size': vocab_size
    }, f)

print('Tokenizer and vocabulary mappings saved.')


In [ ]:
# Find maximum caption length for padding
max_caption_length = max(len(seq) for seq in caption_sequences)
print('Maximum caption length:', max_caption_length)


In [ ]:
# Prepare input-output sequences for caption generation preprocessing
# X_text: partial caption so far
# y_word: next word to predict
# X_image_names: matching image file name for each text sequence

X_text = []
y_word = []
X_image_names = []

for file_name, captions in cleaned_captions_mapping.items():
    for caption in captions:
        seq = tokenizer.texts_to_sequences([caption])[0]

        # Create many training pairs from one caption
        for i in range(1, len(seq)):
            in_seq = seq[:i]
            out_seq = seq[i]
            X_text.append(in_seq)
            y_word.append(out_seq)
            X_image_names.append(file_name)

# Pad input text sequences to same length
X_text = pad_sequences(X_text, maxlen=max_caption_length, padding='post')
y_word = np.array(y_word)
X_image_names = np.array(X_image_names)

print('Text input shape:', X_text.shape)
print('Target output shape:', y_word.shape)
print('Image reference shape:', X_image_names.shape)


In [ ]:
# Show one prepared example
example_index = 0
print('Image file:', X_image_names[example_index])
print('Padded input sequence:', X_text[example_index])
print('Target next word index:', y_word[example_index])
print('Target next word:', index_to_word.get(int(y_word[example_index]), '<unknown>'))


In [ ]:
# Load pretrained InceptionV3 and remove the classification head
base_model = InceptionV3(weights='imagenet')
cnn_model = Model(inputs=base_model.input, outputs=base_model.layers[-2].output)

print('CNN feature extractor ready.')
print('Output feature size:', cnn_model.output_shape)


In [ ]:
# Function to extract one image feature vector
def extract_image_feature(img_path, model):
    img = image.load_img(img_path, target_size=(299, 299))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    feature = model.predict(img_array, verbose=0)
    return feature.flatten()


In [ ]:
# Extract and store one feature vector per image
image_features = {}

for file_name in tqdm(cleaned_captions_mapping.keys()):
    img_path = os.path.join(IMAGES_DIR, file_name)
    if os.path.exists(img_path):
        image_features[file_name] = extract_image_feature(img_path, cnn_model)

print('Extracted features for images:', len(image_features))
first_key = next(iter(image_features))
print('Sample image:', first_key)
print('Sample feature vector shape:', image_features[first_key].shape)


In [ ]:
# Save extracted feature vectors
with open(FEATURES_FILE, 'wb') as f:
    pickle.dump(image_features, f)

print('Image features saved to:', FEATURES_FILE)
